# Filter URLs in CSV

Takes the output of the CSVs from gov search and filters the entries, saving only the ones with:
- `documentType = "guide", "detailed_guide", "answer", "travel_advice", "simple_smart_answer", "transaction", "manual_section"`
- `locale = "en"`
- `withdrawn_at = ""` (empty)

It also removes duplicates within each file (but not globally) by looking at the `content_id`.

In [1]:
# --- Configuration ---
SOURCE_DIR = "search_results"
DEST_DIR = "search_results_filtered"
DOCUMENTTYPE_COLUMN_INDEX = 2
DOCUMENTTYPE_VALID_VALUES = {"guide", "detailed_guide", "answer", "travel_advice", "simple_smart_answer", "transaction", "manual_section"}
LOCALE_COLUMN_INDEX = 4
LOCALE_VALID_VALUES = {"en",}
WITHDRAWNAT_COLUMN_INDEX = 8
CONTENTID_COLUMN_INDEX = 3


In [2]:
import csv
import os
import sys

os.makedirs(DEST_DIR, exist_ok=True)

rows_written_total = 0
duplicates_removed_total = 0

for filename in os.listdir(SOURCE_DIR):
    if not filename.endswith(".csv"):
        continue

    src_path = os.path.join(SOURCE_DIR, filename)
    dst_path = os.path.join(DEST_DIR, filename)
    rows_written = 0
    duplicates_removed = 0
    
    seen = set()

    with open(src_path, newline="") as infile, open(dst_path, "w", newline="") as outfile:
        reader = csv.reader(infile)
        writer = csv.writer(outfile, quoting=csv.QUOTE_MINIMAL)

        header = next(reader, None)
        if header:
            writer.writerow(header)

        for row in reader:
            if row[DOCUMENTTYPE_COLUMN_INDEX] in DOCUMENTTYPE_VALID_VALUES \
            and row[LOCALE_COLUMN_INDEX] in LOCALE_VALID_VALUES \
            and not row[WITHDRAWNAT_COLUMN_INDEX]:
                
                row_key = row[CONTENTID_COLUMN_INDEX]
                
                if row_key not in seen:
                    writer.writerow(row)
                    seen.add(row_key)
                    rows_written += 1
                else:
                    duplicates_removed += 1

    print(f"{filename}: {rows_written} rows written, {duplicates_removed} duplicates removed.")
    rows_written_total = rows_written_total + rows_written
    duplicates_removed_total = duplicates_removed_total + duplicates_removed

print(f"\nTotal rows written: {rows_written_total}, total duplicates removed: {duplicates_removed_total}")

regional_local_gov_taxonomy_branch.csv: 106 rows written, 72 duplicates removed.
government_taxonomy_branch.csv: 544 rows written, 55 duplicates removed.
entering_staying_uk_taxonomy_branch.csv: 239 rows written, 326 duplicates removed.
corporate_information_taxonomy_branch.csv: 95 rows written, 8 duplicates removed.
defence_armed_forces_taxonomy_branch.csv: 290 rows written, 51 duplicates removed.
going_being_abroad_taxonomy_branch.csv: 443 rows written, 86 duplicates removed.
welfare_taxonomy_branch.csv: 185 rows written, 276 duplicates removed.
society_culture_taxonomy_branch.csv: 301 rows written, 66 duplicates removed.
money_taxonomy_branch.csv: 1776 rows written, 797 duplicates removed.
environment_taxonomy_branch.csv: 1244 rows written, 95 duplicates removed.
housing_local_community_taxonomy_branch.csv: 884 rows written, 222 duplicates removed.
education_taxonomy_branch.csv: 461 rows written, 118 duplicates removed.
parenting_child_services_taxonomy_branch.csv: 200 rows written,